TITRE

INTRODUCTION

SOMMAIRE

Installation

In [1]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
#Fonctions:
import src.clear_data as cd
import src.get_data as gd

Préparation des données

Adresses

In [2]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

Import et modifications/nettoyage des données

In [3]:
#Import
df_dvf_raw = gd.fetch_dvf_api()
gdf_metro_raw = gd.fetch_metro_api()
#Nettoyage
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)
gdf_final['prix_m2'] = gdf_final['valeur_fonciere'] / gdf_final['surface_reelle_bati']
print(gdf_final.head)
print(gdf_final.columns)

--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération des données Métro via API (Rennes Métropole) ---
<bound method NDFrame.head of         date_mutation nature_mutation  valeur_fonciere code_commune  \
1365203    2023-01-04           Vente         160000.0        35238   
1365206    2023-01-07           Vente         320000.0        35238   
1365214    2023-01-06           Vente         120000.0        35238   
1365215    2023-01-06           Vente         120000.0        35238   
1365243    2023-01-04           Vente         310000.0        35238   
...               ...             ...              ...          ...   
1427961    2023-12-20           Vente         180000.0        35238   
1428108    2023-12-22           Vente         151480.0        35238   
1428292    2023-11-17           Vente         263000.0        35238   
1428428    2023-11-14           Vente         270000.0        35238   
1428450    2023-08-30           Vente         250000.0        35238   

        nom_commune   type_local  surface_reelle_ba

In [24]:
import geopandas as gpd
import pandas as pd

Importation des deux bases de données :
- Demandes de valeurs foncières
- Topologie des points d'arrêt de métro du réseau STAR

Points d'arrêt de métro 

In [25]:
# URL directe vers l'export GeoJSON du portail Open Data
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

# Lecture directe depuis l'URL
gdf_metro = gpd.read_file(url_metro)

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(gdf_metro.head())
print(gdf_metro.crs)

  codeinseecommune                            coordonnees  \
0            35281  [48.0922543436006, -1.70335007013448]   
1            35238   [48.105036267316, -1.69241228789686]   
2            35238  [48.1042189200449, -1.67189613885799]   
3            35051  [48.1314259119796, -1.62024738680177]   
4            35051  [48.1271194329088, -1.62826328652191]   

  stop_id_station_parente adressenumero               adressevoie  \
0                 6-15064           1 B         rue Louis Braille   
1                 6-15061            15               rue Surcouf   
2                 6-15008            15          place de la Gare   
3                 6-15050            14               la Rochelle   
4                 6-15051             1  avenue de Belle Fontaine   

  estaccessiblepmr      nomstationparente stop_id    id  \
0             true  Saint-Jacques - Gaîté  6-5065  5065   
1             true               Mabilais  6-5068  5068   
2             true                  Gares

In [ ]:
print(gdf_metro.columns)


Demandes de Valeurs Foncières (DVF)

In [26]:
# Exemple pour l'année 2023 (ou la plus récente disponible sur l'API DVF)
# URL des fichiers consolidés par Etalab
url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

# On précise le séparateur (souvent la virgule) et on gère la compression .gz
# précisez 'low_memory=False' pour éviter les warnings sur les types
df_dvf = pd.read_csv(url_dvf, compression='gzip', sep=',', low_memory=False)

# Pour filtrer sur Rennes (code commune 35238)
df_rennes = df_dvf[df_dvf['code_commune'] == '35238'].copy()

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(df_rennes.head())

         id_mutation date_mutation  numero_disposition nature_mutation  \
1365203  2023-471776    2023-01-04                   1           Vente   
1365204  2023-471776    2023-01-04                   1           Vente   
1365205  2023-471777    2023-01-07                   1           Vente   
1365206  2023-471777    2023-01-07                   1           Vente   
1365207  2023-471777    2023-01-07                   1           Vente   

         valeur_fonciere  adresse_numero adresse_suffixe adresse_nom_voie  \
1365203         160000.0            15.0             NaN       BD D ANJOU   
1365204         160000.0            15.0             NaN       BD D ANJOU   
1365205         320000.0           173.0             NaN  RUE DE FOUGERES   
1365206         320000.0           173.0             NaN  RUE DE FOUGERES   
1365207         320000.0           173.0             NaN  RUE DE FOUGERES   

        adresse_code_voie  code_postal  ...   type_local surface_reelle_bati  \
1365203     

In [ ]:
print(df_rennes.columns)

Modèle de prédiction du prix :

In [9]:
from src.model import preparer_et_entrainer_did, predire_impact_nouvelle_station

# Entraînement
model_did, features = preparer_et_entrainer_did(gdf_final)

# Scénario : Un bien à 2km d'une station vs 100m
bien_temoin = {
    'nombre_pieces_principales': 3, 'surface_terrain': 200,
    'dist_metro_A': 2000, 'dist_metro_B': 5000, 'proche_station_A': 0,
    'proche_station_B': 0
}

bien_traite = bien_temoin.copy()
bien_traite.update({'dist_metro_A': 100, 'proche_station_A': 1, 'proche_station_B': 0})

prix_temoin = predire_impact_nouvelle_station(model_did, bien_temoin)
prix_traite = predire_impact_nouvelle_station(model_did, bien_traite)

print(f"Effet estimé de la station : {prix_traite - prix_temoin:.2f} €/m²")

Effet estimé de la station : -1239.66 €/m²


In [ ]:

import pandas as pd
importances = pd.DataFrame({'feature': features, 'importance': model_did.feature_importances_})
print(importances.sort_values(by='importance', ascending=False))

Les variables de distance continue capte la majeur partie de l'info. Les variables binaire de proximité avec station A ou B semble inutile. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simuler les prix pour des distances de 0 à 3000 mètres
distances = np.linspace(0, 3000, 50)
prix_simules = [predire_impact_nouvelle_station(model_did, {'nombre_pieces_principales': 3, 'surface_terrain': 200, 'dist_metro_A': d, 'dist_metro_B': 5000, 'proche_station_A': 1 if d < 800 else 0, 'proche_station_B': 0}) for d in distances]

plt.plot(distances, prix_simules)
plt.title("Évolution du prix en fonction de la distance au métro")
plt.xlabel("Distance au métro (m)")
plt.ylabel("Prix estimé (€/m²)")
plt.show()